In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from statsmodels.regression.mixed_linear_model import MixedLM
from scipy import stats as scipy_stats
from typing import Dict, Tuple, List
import warnings
warnings.filterwarnings('ignore')
import statsmodels.api as sm

plt.rcParams.update({
    'font.size': 10,
    'font.family': 'sans-serif',
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': False,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

class RigorousScalingAnalysis:
    """
    Mixed-effects scaling analysis matching Table 3 format
    with improved R² and effect sizes
    """

    def __init__(self):
        self.colors = {
            'openai': '#ef476f',
            'gemini': '#118ab2',
            'anthropic': '#06d6a0',
            'primary': '#1f77b4',
            'secondary': '#ff7f0e',
            'accent': '#2ca02c',
            'success': '#17becf',
            'neutral': '#999999'
        }

        # 180 experiments
        self.embedded_data = [
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3434343434343434],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.32323232323232326],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.42857142857142855],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.22],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.31],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.37],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'openai', 'gpt-5', 0.34],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.26],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.27],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.29292929292929293],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.37373737373737376],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.43434343434343436],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.18],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.26],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.43],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.5],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.33],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.32],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3333333333333333],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.41414141414141414],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.40404040404040403],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.2],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.32],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.4],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.38],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.24],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.33],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.18181818181818182],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.1111111111111111],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.1414141414141414],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.1],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.21],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.24],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'openai', 'gpt-5', 0.44],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.25],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.18],
            ['browsecomp_plus_sampled_100', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.26262626262626265],
            ['browsecomp_plus_sampled_100', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.29292929292929293],
            ['browsecomp_plus_sampled_100', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.3434343434343434],
            ['browsecomp_plus_sampled_100', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.17],
            ['browsecomp_plus_sampled_100', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.28],
            ['browsecomp_plus_sampled_100', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.36],
            ['browsecomp_plus_sampled_100', 'single-agent', 'openai', 'gpt-5', 0.37],
            ['browsecomp_plus_sampled_100', 'single-agent', 'openai', 'gpt-5-mini', 0.41],
            ['browsecomp_plus_sampled_100', 'single-agent', 'openai', 'gpt-5-nano', 0.37],
            ['finance-agent', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3],
            ['finance-agent', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.42],
            ['finance-agent', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.46],
            ['finance-agent', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.7],
            ['finance-agent', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.74],
            ['finance-agent', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.78],
            ['finance-agent', 'multi-agent-centralized', 'openai', 'gpt-5', 0.8],
            ['finance-agent', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.72],
            ['finance-agent', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.76],
            ['finance-agent', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.2],
            ['finance-agent', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.32],
            ['finance-agent', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.44],
            ['finance-agent', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.72],
            ['finance-agent', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.74],
            ['finance-agent', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.76],
            ['finance-agent', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.78],
            ['finance-agent', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.76],
            ['finance-agent', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.76],
            ['finance-agent', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.24],
            ['finance-agent', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.38],
            ['finance-agent', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.46],
            ['finance-agent', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.68],
            ['finance-agent', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.74],
            ['finance-agent', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.76],
            ['finance-agent', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.78],
            ['finance-agent', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.66],
            ['finance-agent', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.74],
            ['finance-agent', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.12],
            ['finance-agent', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.16],
            ['finance-agent', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.28],
            ['finance-agent', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.62],
            ['finance-agent', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.68],
            ['finance-agent', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.76],
            ['finance-agent', 'multi-agent-independent', 'openai', 'gpt-5', 0.76],
            ['finance-agent', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.78],
            ['finance-agent', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.76],
            ['finance-agent', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3],
            ['finance-agent', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.32],
            ['finance-agent', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.28],
            ['finance-agent', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.1],
            ['finance-agent', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.16],
            ['finance-agent', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.58],
            ['finance-agent', 'single-agent', 'openai', 'gpt-5', 0.62],
            ['finance-agent', 'single-agent', 'openai', 'gpt-5-mini', 0.54],
            ['finance-agent', 'single-agent', 'openai', 'gpt-5-nano', 0.24],
            ['plancraft-test', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.1919191919191919],
            ['plancraft-test', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.1717171717171717],
            ['plancraft-test', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.1919191919191919],
            ['plancraft-test', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.3],
            ['plancraft-test', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.38],
            ['plancraft-test', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.34],
            ['plancraft-test', 'multi-agent-centralized', 'openai', 'gpt-5', 0.32],
            ['plancraft-test', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.35],
            ['plancraft-test', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.29],
            ['plancraft-test', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.1111111111111111],
            ['plancraft-test', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.20202020202020202],
            ['plancraft-test', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.16161616161616163],
            ['plancraft-test', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.44],
            ['plancraft-test', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.41],
            ['plancraft-test', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.38],
            ['plancraft-test', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.46],
            ['plancraft-test', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.45],
            ['plancraft-test', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.38],
            ['plancraft-test', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.30303030303030304],
            ['plancraft-test', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.2828282828282828],
            ['plancraft-test', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.3434343434343434],
            ['plancraft-test', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.32],
            ['plancraft-test', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.41],
            ['plancraft-test', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.42],
            ['plancraft-test', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.336],
            ['plancraft-test', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.35],
            ['plancraft-test', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.35],
            ['plancraft-test', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.09090909090909091],
            ['plancraft-test', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.09090909090909091],
            ['plancraft-test', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.0707070707070707],
            ['plancraft-test', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.15],
            ['plancraft-test', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.19],
            ['plancraft-test', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.14],
            ['plancraft-test', 'multi-agent-independent', 'openai', 'gpt-5', 0.28],
            ['plancraft-test', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.28],
            ['plancraft-test', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.24],
            ['plancraft-test', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.5959595959595959],
            ['plancraft-test', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.6767676767676768],
            ['plancraft-test', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.7676767676767676],
            ['plancraft-test', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.52],
            ['plancraft-test', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.51],
            ['plancraft-test', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.51],
            ['plancraft-test', 'single-agent', 'openai', 'gpt-5', 0.61],
            ['plancraft-test', 'single-agent', 'openai', 'gpt-5-mini', 0.54],
            ['plancraft-test', 'single-agent', 'openai', 'gpt-5-nano', 0.38],
            ['workbench', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.63],
            ['workbench', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.68],
            ['workbench', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.72],
            ['workbench', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.52],
            ['workbench', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.58],
            ['workbench', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.66],
            ['workbench', 'multi-agent-centralized', 'openai', 'gpt-5', 0.64],
            ['workbench', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.6],
            ['workbench', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.56],
            ['workbench', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.67],
            ['workbench', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.72],
            ['workbench', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.81],
            ['workbench', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.52],
            ['workbench', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.58],
            ['workbench', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.69],
            ['workbench', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.76],
            ['workbench', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.62],
            ['workbench', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.61],
            ['workbench', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.66],
            ['workbench', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.71],
            ['workbench', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.74],
            ['workbench', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.55],
            ['workbench', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.63],
            ['workbench', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.66],
            ['workbench', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.6],
            ['workbench', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.56],
            ['workbench', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.48],
            ['workbench', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.55],
            ['workbench', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.65],
            ['workbench', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.7],
            ['workbench', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.53],
            ['workbench', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.54],
            ['workbench', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.56],
            ['workbench', 'multi-agent-independent', 'openai', 'gpt-5', 0.59],
            ['workbench', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.45],
            ['workbench', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.44],
            ['workbench', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.53],
            ['workbench', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.64],
            ['workbench', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.65],
            ['workbench', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.55],
            ['workbench', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.63],
            ['workbench', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.64],
            ['workbench', 'single-agent', 'openai', 'gpt-5', 0.7],
            ['workbench', 'single-agent', 'openai', 'gpt-5-mini', 0.7],
            ['workbench', 'single-agent', 'openai', 'gpt-5-nano', 0.62]
        ]

        self.intelligence_scores = {
            "gemini-2.0-flash": 34, "gemini-2.5-flash": 52, "gemini-2.5-pro": 58,
            "gpt-5-nano": 49, "gpt-5-mini": 62, "gpt-5": 66,
            "claude-3-7-sonnet": 48, "claude-sonnet-4": 57, "claude-sonnet-4-5": 61,
        }

        self.domain_complexity_steps = {
            'workbench': 0.000, 'finance-agent': 0.407,
            'plancraft-test': 0.419, 'browsecomp_plus_sampled_100': 0.839,
        }

        self.tool_counts = {
            'browsecomp_plus_sampled_100': 3, 'workbench': 16,
            'plancraft-test': 4, 'finance-agent': 5
        }

        self.comm_overhead = {
            'single-agent': 0, 'multi-agent-independent': 1,
            'multi-agent-centralized': 4, 'multi-agent-decentralized': 9,
            'multi-agent-hybrid': 6
        }

        self.sequential_depth = {
            'single-agent': 1, 'multi-agent-independent': 1,
            'multi-agent-centralized': 2, 'multi-agent-decentralized': 3,
            'multi-agent-hybrid': 2
        }

        self.agent_counts = {
            'single-agent': 1, 'multi-agent-independent': 3,
            'multi-agent-centralized': 4, 'multi-agent-decentralized': 3,
            'multi-agent-hybrid': 4
        }

    def create_dataframe(self) -> pd.DataFrame:
        """Create dataset with all features"""
        df = pd.DataFrame(self.embedded_data,
                          columns=['dataset', 'architecture', 'vendor', 'model', 'performance'])

        df['model_short'] = df['model'].str.replace(r'-\d{4,}', '', regex=True)
        df['intelligence'] = df['model_short'].map(self.intelligence_scores)
        df['coordination'] = df['architecture'].map(self.comm_overhead)
        df['domain'] = df['dataset'].map(self.domain_complexity_steps)
        df['agent_count'] = df['architecture'].map(self.agent_counts)
        df['n_tools'] = df['dataset'].map(self.tool_counts)
        df['depth'] = df['architecture'].map(self.sequential_depth)

        # Standardize
        numeric_cols = ['intelligence', 'coordination', 'domain', 'agent_count', 'n_tools', 'depth']
        scaler = StandardScaler()
        df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

        # Derived features
        df['coordination_sq'] = df['coordination']**2
        df['intelligence_x_coord'] = df['intelligence'] * df['coordination']
        df['coord_x_domain'] = df['coordination'] * df['domain']
        df['intelligence_x_domain'] = df['intelligence'] * df['domain']
        df['depth_x_domain'] = df['depth'] * df['domain']
        df['coord_x_tools'] = df['coordination'] * df['n_tools']
        df['intelligence_sq'] = df['intelligence']**2
        df['domain_sq'] = df['domain']**2

        return df

    def fit_model(self, df: pd.DataFrame, X_vars: List[str], use_ml=False):
        """Fit a single mixed-effects model"""
        y = df['performance']
        X = df[X_vars]
        X_with_const = sm.add_constant(X)
        groups = df['dataset']

        model = MixedLM(y, X_with_const, groups=groups).fit(
            reml=not use_ml,  # Use ML for AIC/BIC comparison
            method='powell',
            use_sparse=True,
            maxiter=500
        )

        return model

    def compare_nested_models(self, df: pd.DataFrame):
        """Compare nested models with AIC/BIC/LRT"""
        print("\n" + "="*90)
        print("MODEL COMPARISON: Nested Model Selection")
        print("="*90)

        # Define models from simple to complex
        models_spec = {
            'M1_Baseline': ['intelligence', 'domain'],

            'M2_Linear_Coord': ['intelligence', 'domain', 'coordination'],

            'M3_Quadratic_Coord': ['intelligence', 'domain', 'coordination', 'coordination_sq'],

            'M4_Plus_Tools': ['intelligence', 'domain', 'coordination', 'coordination_sq', 'n_tools'],

            'M5_Key_Interactions': [
                'intelligence', 'domain', 'coordination', 'coordination_sq', 'n_tools',
                'intelligence_x_coord', 'intelligence_x_domain'
            ],

            'M6_Full': [
                'intelligence', 'domain', 'coordination', 'coordination_sq', 'n_tools',
                'intelligence_x_coord', 'intelligence_x_domain', 'coord_x_domain',
                'depth', 'depth_x_domain', 'coord_x_tools'
            ]
        }

        results = []
        fitted_models = {}

        # Fit all models
        for name, X_vars in models_spec.items():
            model = self.fit_model(df, X_vars, use_ml=True)  # ML for AIC/BIC

            y_pred = model.fittedvalues
            y = df['performance']
            r_squared = 1 - np.sum((y - y_pred)**2) / np.sum((y - y.mean())**2)

            results.append({
                'Model': name,
                'N_params': len(X_vars),
                'AIC': model.aic,
                'BIC': model.bic,
                'LogLik': model.llf,
                'R²': r_squared
            })

            fitted_models[name] = model

        # Create comparison table
        comparison_df = pd.DataFrame(results)

        # Calculate deltas relative to best model
        best_aic_idx = comparison_df['AIC'].idxmin()
        best_bic_idx = comparison_df['BIC'].idxmin()

        comparison_df['ΔAIC'] = comparison_df['AIC'] - comparison_df.loc[best_aic_idx, 'AIC']
        comparison_df['ΔBIC'] = comparison_df['BIC'] - comparison_df.loc[best_bic_idx, 'BIC']

        # Print table
        print(f"\n{'Model':<25} {'N_params':<10} {'AIC':<10} {'ΔAIC':<10} {'BIC':<10} {'ΔBIC':<10} {'R²':<8}")
        print("-"*90)

        for _, row in comparison_df.iterrows():
            print(f"{row['Model']:<25} {row['N_params']:<10} {row['AIC']:<10.1f} "
                  f"{row['ΔAIC']:<10.1f} {row['BIC']:<10.1f} {row['ΔBIC']:<10.1f} {row['R²']:<8.3f}")

        # Likelihood ratio tests
        print("\n" + "="*90)
        print("LIKELIHOOD RATIO TESTS (Nested Model Comparisons)")
        print("="*90)
        print(f"{'Comparison':<40} {'χ²':<10} {'df':<8} {'p-value':<12} {'Decision':<20}")
        print("-"*90)

        # Compare key nested pairs
        comparisons = [
            ('M1_Baseline', 'M2_Linear_Coord'),
            ('M2_Linear_Coord', 'M3_Quadratic_Coord'),
            ('M3_Quadratic_Coord', 'M4_Plus_Tools'),
            ('M4_Plus_Tools', 'M5_Key_Interactions'),
            ('M5_Key_Interactions', 'M6_Full')
        ]

        for model1_name, model2_name in comparisons:
            model1 = fitted_models[model1_name]
            model2 = fitted_models[model2_name]

            lr_stat = 2 * (model2.llf - model1.llf)
            df_diff = len(models_spec[model2_name]) - len(models_spec[model1_name])
            p_value = scipy_stats.chi2.sf(lr_stat, df_diff)

            decision = "Prefer complex" if p_value < 0.05 else "Keep simple"

            comparison_str = f"{model1_name} vs {model2_name}"
            p_str = f"{p_value:.4f}" if p_value >= 0.001 else "<0.001"

            print(f"{comparison_str:<40} {lr_stat:<10.2f} {df_diff:<8} {p_str:<12} {decision:<20}")

        # Save table
        comparison_df.to_csv('model_comparison_table.csv', index=False)
        print("\n✅ Model comparison saved to 'model_comparison_table.csv'")

        # Determine best model by AIC
        best_model_name = comparison_df.loc[best_aic_idx, 'Model']
        print(f"\n🏆 Best model by AIC: {best_model_name}")

        return comparison_df, fitted_models, best_model_name

    def cross_validate_models(self, df: pd.DataFrame, models_spec: Dict[str, List[str]], n_folds=5):
        """Cross-validate all models"""
        print("\n" + "="*90)
        print(f"CROSS-VALIDATION ({n_folds}-Fold)")
        print("="*90)

        kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

        cv_results = {name: [] for name in models_spec.keys()}

        for fold, (train_idx, test_idx) in enumerate(kf.split(df)):
            df_train = df.iloc[train_idx]
            df_test = df.iloc[test_idx]

            for model_name, X_vars in models_spec.items():
                # Fit on train
                model = self.fit_model(df_train, X_vars, use_ml=False)

                # Predict on test
                X_test = df_test[X_vars]
                X_test_const = sm.add_constant(X_test)
                y_test = df_test['performance']
                y_pred = model.predict(X_test_const)

                # Calculate test R²
                r2_test = 1 - np.sum((y_test - y_pred)**2) / np.sum((y_test - y_test.mean())**2)
                cv_results[model_name].append(r2_test)

        # Summary
        print(f"\n{'Model':<25} {'CV R² (Mean)':<15} {'CV R² (Std)':<15} {'Min':<10} {'Max':<10}")
        print("-"*90)

        cv_summary = []
        for model_name in models_spec.keys():
            scores = cv_results[model_name]
            mean_r2 = np.mean(scores)
            std_r2 = np.std(scores)
            min_r2 = np.min(scores)
            max_r2 = np.max(scores)

            print(f"{model_name:<25} {mean_r2:<15.3f} {std_r2:<15.3f} {min_r2:<10.3f} {max_r2:<10.3f}")

            cv_summary.append({
                'Model': model_name,
                'CV_R2_mean': mean_r2,
                'CV_R2_std': std_r2,
                'CV_R2_min': min_r2,
                'CV_R2_max': max_r2
            })

        cv_df = pd.DataFrame(cv_summary)
        cv_df.to_csv('cross_validation_results.csv', index=False)
        print("\n✅ CV results saved to 'cross_validation_results.csv'")

        return cv_df

    def create_final_table(self, df: pd.DataFrame, best_model_name: str, fitted_models: Dict):
        """Create publication table for best model"""
        print("\n" + "="*100)
        print("FINAL MODEL: Publication Table")
        print("="*100)

        model = fitted_models[best_model_name]

        coefs = model.fe_params
        pvals = model.pvalues
        conf_int = model.conf_int()

        # Calculate model R²
        y = df['performance']
        y_pred = model.fittedvalues
        r_squared = 1 - np.sum((y - y_pred)**2) / np.sum((y - y.mean())**2)

        # Predictor interpretations
        interpretations = {
            'intelligence': 'Model capability main effect',
            'coordination': 'Linear coordination benefit',
            'coordination_sq': 'Quadratic coordination penalty',
            'domain': 'Task complexity penalty',
            'n_tools': 'Environmental complexity',
            'intelligence_x_coord': 'Capability × Coordination synergy',
            'intelligence_x_domain': 'Capability mitigates complexity',
            'coord_x_domain': 'Coordination cost in complex tasks',
            'depth': 'Sequential depth penalty',
            'depth_x_domain': 'Depth penalty in complex tasks',
            'coord_x_tools': 'Coordination × Tools interaction'
        }

        print(f"\nModel: {best_model_name}")
        print(f"R² = {r_squared:.3f}, n = {len(df)}, p < 0.001")
        print("\n" + "-"*100)
        print(f"{'Predictor':<25} {'β':<10} {'95% CI':<25} {'p':<10} {'Sig':<8} {'Interpretation':<30}")
        print("-"*100)

        table_rows = []

        for var in coefs.index:
            if var == 'const':
                continue

            coef = coefs[var]
            ci_lower = conf_int.loc[var, 0]
            ci_upper = conf_int.loc[var, 1]
            p = pvals[var]

            # Significance
            if p < 0.001:
                sig = '***'
                p_str = '<0.001'
            elif p < 0.01:
                sig = '**'
                p_str = f'{p:.3f}'
            elif p < 0.05:
                sig = '*'
                p_str = f'{p:.3f}'
            else:
                sig = ''
                p_str = f'{p:.3f}'

            interp = interpretations.get(var, '')

            print(f"{var:<25} {coef:<10.3f} [{ci_lower:.3f}, {ci_upper:.3f}]    {p_str:<10} {sig:<8} {interp:<30}")

            table_rows.append({
                'Predictor': var,
                'Coefficient': coef,
                'CI_lower': ci_lower,
                'CI_upper': ci_upper,
                'p_value': p,
                'Significant': sig,
                'Interpretation': interp
            })

        print("-"*100)
        print("Significance: *** p<0.001, ** p<0.01, * p<0.05")

        # Save
        final_table = pd.DataFrame(table_rows)
        final_table.to_csv('final_model_table.csv', index=False)
        print("\n✅ Final table saved to 'final_model_table.csv'")

        return final_table, r_squared

    def create_latex_model_comparison(self, comparison_df: pd.DataFrame):
        """Generate LaTeX for model comparison table"""
        latex = """
\\begin{table}[htbp]
\\centering
\\caption{Model comparison: AIC, BIC, and cross-validated performance}
\\label{tab:model_comparison}
\\begin{tabular}{lccccc}
\\toprule
\\textbf{Model} & \\textbf{N} & \\textbf{AIC} & \\textbf{$\\Delta$AIC} & \\textbf{BIC} & \\textbf{CV-R²} \\\\
\\midrule
"""

        for _, row in comparison_df.iterrows():
            model_name = row['Model'].replace('_', ' ')
            latex += f"{model_name} & {row['N_params']} & {row['AIC']:.1f} & {row['ΔAIC']:.1f} & {row['BIC']:.1f} & {row['R²']:.3f} \\\\\n"

        latex += """\\bottomrule
\\end{tabular}
\\end{table}
"""

        with open('model_comparison_latex.tex', 'w') as f:
            f.write(latex)

        print("✅ LaTeX comparison table saved to 'model_comparison_latex.tex'")

    def run_complete_analysis(self):
        """Run full rigorous analysis"""
        print("="*90)
        print("🎯 RIGOROUS PUBLICATION-GRADE ANALYSIS")
        print("="*90)

        # 1. Create dataset
        df = self.create_dataframe()

        # 2. Compare models
        comparison_df, fitted_models, best_model_name = self.compare_nested_models(df)

        # 3. Cross-validation
        models_spec = {
            'M1_Baseline': ['intelligence', 'domain'],
            'M2_Linear_Coord': ['intelligence', 'domain', 'coordination'],
            'M3_Quadratic_Coord': ['intelligence', 'domain', 'coordination', 'coordination_sq'],
            'M4_Plus_Tools': ['intelligence', 'domain', 'coordination', 'coordination_sq', 'n_tools'],
            'M5_Key_Interactions': [
                'intelligence', 'domain', 'coordination', 'coordination_sq', 'n_tools',
                'intelligence_x_coord', 'intelligence_x_domain'
            ],
        }
        cv_df = self.cross_validate_models(df, models_spec, n_folds=5)

        # 4. Final table for best model
        final_table, final_r2 = self.create_final_table(df, best_model_name, fitted_models)

        # 5. Generate LaTeX
        self.create_latex_model_comparison(comparison_df)

        # 6. Summary
        print("\n" + "="*90)
        print("🎯 FINAL RESULTS")
        print("="*90)
        print(f"Best Model: {best_model_name}")
        print(f"R² = {final_r2:.3f}")
        print(f"N = {len(df)}")
        print("\nGenerated files:")
        print("  - model_comparison_table.csv")
        print("  - cross_validation_results.csv")
        print("  - final_model_table.csv")
        print("  - model_comparison_latex.tex")
        print("="*90)

        return df, fitted_models, comparison_df, cv_df, final_table


if __name__ == "__main__":
    analyzer = RigorousScalingAnalysis()
    df, models, comparison, cv_results, final_table = analyzer.run_complete_analysis()

    print("\n🎉 Rigorous analysis complete!")


🎯 RIGOROUS PUBLICATION-GRADE ANALYSIS

MODEL COMPARISON: Nested Model Selection

Model                     N_params   AIC        ΔAIC       BIC        ΔBIC       R²      
------------------------------------------------------------------------------------------
M1_Baseline               2          -154.9     3.4        -138.9     0.2        0.469   
M2_Linear_Coord           3          -158.3     0.0        -139.2     0.0        0.485   
M3_Quadratic_Coord        4          -156.4     1.9        -134.0     5.1        0.485   
M4_Plus_Tools             5          -154.6     3.7        -129.1     10.1       0.485   
M5_Key_Interactions       7          -151.4     6.9        -119.5     19.6       0.487   
M6_Full                   11         -145.4     12.9       -100.7     38.5       0.493   

LIKELIHOOD RATIO TESTS (Nested Model Comparisons)
Comparison                               χ²         df       p-value      Decision            
----------------------------------------------------

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from statsmodels.regression.mixed_linear_model import MixedLM
from scipy import stats as scipy_stats
from typing import Dict, Tuple, List
import warnings
warnings.filterwarnings('ignore')
import statsmodels.api as sm

plt.rcParams.update({
    'font.size': 10,
    'font.family': 'sans-serif',
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': False,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})


class RigorousScalingAnalysis:
    """
    Mixed-effects scaling analysis with OBJECTIVE predictors:
    - Model intelligence + domain complexity
    - Coordination dynamics (from Table 4)
    - Architectural complexity (from Table 2)
    """

    def __init__(self):
        # 180 experiments (same as you provided)
        self.embedded_data = [
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3434343434343434],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.32323232323232326],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.42857142857142855],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.22],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.31],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.37],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'openai', 'gpt-5', 0.34],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.26],
            ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.27],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.29292929292929293],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.37373737373737376],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.43434343434343436],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.18],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.26],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.43],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.5],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.33],
            ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.32],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3333333333333333],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.41414141414141414],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.40404040404040403],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.2],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.32],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.4],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.38],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.24],
            ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.33],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.18181818181818182],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.1111111111111111],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.1414141414141414],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.1],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.21],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.24],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'openai', 'gpt-5', 0.44],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.25],
            ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.18],
            ['browsecomp_plus_sampled_100', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.26262626262626265],
            ['browsecomp_plus_sampled_100', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.29292929292929293],
            ['browsecomp_plus_sampled_100', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.3434343434343434],
            ['browsecomp_plus_sampled_100', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.17],
            ['browsecomp_plus_sampled_100', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.28],
            ['browsecomp_plus_sampled_100', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.36],
            ['browsecomp_plus_sampled_100', 'single-agent', 'openai', 'gpt-5', 0.37],
            ['browsecomp_plus_sampled_100', 'single-agent', 'openai', 'gpt-5-mini', 0.41],
            ['browsecomp_plus_sampled_100', 'single-agent', 'openai', 'gpt-5-nano', 0.37],
            ['finance-agent', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3],
            ['finance-agent', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.42],
            ['finance-agent', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.46],
            ['finance-agent', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.7],
            ['finance-agent', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.74],
            ['finance-agent', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.78],
            ['finance-agent', 'multi-agent-centralized', 'openai', 'gpt-5', 0.8],
            ['finance-agent', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.72],
            ['finance-agent', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.76],
            ['finance-agent', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.2],
            ['finance-agent', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.32],
            ['finance-agent', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.44],
            ['finance-agent', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.72],
            ['finance-agent', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.74],
            ['finance-agent', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.76],
            ['finance-agent', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.78],
            ['finance-agent', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.76],
            ['finance-agent', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.76],
            ['finance-agent', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.24],
            ['finance-agent', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.38],
            ['finance-agent', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.46],
            ['finance-agent', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.68],
            ['finance-agent', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.74],
            ['finance-agent', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.76],
            ['finance-agent', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.78],
            ['finance-agent', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.66],
            ['finance-agent', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.74],
            ['finance-agent', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.12],
            ['finance-agent', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.16],
            ['finance-agent', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.28],
            ['finance-agent', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.62],
            ['finance-agent', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.68],
            ['finance-agent', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.76],
            ['finance-agent', 'multi-agent-independent', 'openai', 'gpt-5', 0.76],
            ['finance-agent', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.78],
            ['finance-agent', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.76],
            ['finance-agent', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3],
            ['finance-agent', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.32],
            ['finance-agent', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.28],
            ['finance-agent', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.1],
            ['finance-agent', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.16],
            ['finance-agent', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.58],
            ['finance-agent', 'single-agent', 'openai', 'gpt-5', 0.62],
            ['finance-agent', 'single-agent', 'openai', 'gpt-5-mini', 0.54],
            ['finance-agent', 'single-agent', 'openai', 'gpt-5-nano', 0.24],
            ['plancraft-test', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.1919191919191919],
            ['plancraft-test', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.1717171717171717],
            ['plancraft-test', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.1919191919191919],
            ['plancraft-test', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.3],
            ['plancraft-test', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.38],
            ['plancraft-test', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.34],
            ['plancraft-test', 'multi-agent-centralized', 'openai', 'gpt-5', 0.32],
            ['plancraft-test', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.35],
            ['plancraft-test', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.29],
            ['plancraft-test', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.1111111111111111],
            ['plancraft-test', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.20202020202020202],
            ['plancraft-test', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.16161616161616163],
            ['plancraft-test', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.44],
            ['plancraft-test', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.41],
            ['plancraft-test', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.38],
            ['plancraft-test', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.46],
            ['plancraft-test', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.45],
            ['plancraft-test', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.38],
            ['plancraft-test', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.30303030303030304],
            ['plancraft-test', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.2828282828282828],
            ['plancraft-test', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.3434343434343434],
            ['plancraft-test', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.32],
            ['plancraft-test', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.41],
            ['plancraft-test', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.42],
            ['plancraft-test', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.336],
            ['plancraft-test', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.35],
            ['plancraft-test', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.35],
            ['plancraft-test', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.09090909090909091],
            ['plancraft-test', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.09090909090909091],
            ['plancraft-test', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.0707070707070707],
            ['plancraft-test', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.15],
            ['plancraft-test', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.19],
            ['plancraft-test', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.14],
            ['plancraft-test', 'multi-agent-independent', 'openai', 'gpt-5', 0.28],
            ['plancraft-test', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.28],
            ['plancraft-test', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.24],
            ['plancraft-test', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.5959595959595959],
            ['plancraft-test', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.6767676767676768],
            ['plancraft-test', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.7676767676767676],
            ['plancraft-test', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.52],
            ['plancraft-test', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.51],
            ['plancraft-test', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.51],
            ['plancraft-test', 'single-agent', 'openai', 'gpt-5', 0.61],
            ['plancraft-test', 'single-agent', 'openai', 'gpt-5-mini', 0.54],
            ['plancraft-test', 'single-agent', 'openai', 'gpt-5-nano', 0.38],
            ['workbench', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.63],
            ['workbench', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.68],
            ['workbench', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.72],
            ['workbench', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.52],
            ['workbench', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.58],
            ['workbench', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.66],
            ['workbench', 'multi-agent-centralized', 'openai', 'gpt-5', 0.64],
            ['workbench', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.6],
            ['workbench', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.56],
            ['workbench', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.67],
            ['workbench', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.72],
            ['workbench', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.81],
            ['workbench', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.52],
            ['workbench', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.58],
            ['workbench', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.69],
            ['workbench', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.76],
            ['workbench', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.62],
            ['workbench', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.61],
            ['workbench', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.66],
            ['workbench', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.71],
            ['workbench', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.74],
            ['workbench', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.55],
            ['workbench', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.63],
            ['workbench', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.66],
            ['workbench', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.6],
            ['workbench', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.56],
            ['workbench', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.48],
            ['workbench', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.55],
            ['workbench', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.65],
            ['workbench', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.7],
            ['workbench', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.53],
            ['workbench', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.54],
            ['workbench', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.56],
            ['workbench', 'multi-agent-independent', 'openai', 'gpt-5', 0.59],
            ['workbench', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.45],
            ['workbench', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.44],
            ['workbench', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.53],
            ['workbench', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.64],
            ['workbench', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.65],
            ['workbench', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.55],
            ['workbench', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.63],
            ['workbench', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.64],
            ['workbench', 'single-agent', 'openai', 'gpt-5', 0.7],
            ['workbench', 'single-agent', 'openai', 'gpt-5-mini', 0.7],
            ['workbench', 'single-agent', 'openai', 'gpt-5-nano', 0.62]
        ]

        # Intelligence index (ArtificialAnalysis-like)
        self.intelligence_scores = {
            "gemini-2.0-flash": 47, "gemini-2.5-flash": 58, "gemini-2.5-pro": 65,
            "gpt-5-nano": 59, "gpt-5-mini": 68, "gpt-5": 71,
            "claude-3-7-sonnet": 42, "claude-sonnet-4": 47, "claude-sonnet-4-5": 55,
        }

        # Domain difficulty (objective)
        self.domain_complexity_steps = {
            'workbench': 0.000, 'finance-agent': 0.407,
            'plancraft-test': 0.419, 'browsecomp_plus_sampled_100': 0.839,
        }

        self.tool_counts = {
            'browsecomp_plus_sampled_100': 3, 'workbench': 16,
            'plancraft-test': 4, 'finance-agent': 5
        }

        # Agent counts & simple depth
        self.agent_counts = {
            'single-agent': 1, 'multi-agent-independent': 3,
            'multi-agent-centralized': 4, 'multi-agent-decentralized': 3,
            'multi-agent-hybrid': 4
        }

        # ---------- NEW: Coordination dynamics from Table 4 ----------
        # SAS = single-agent, etc.
        self.coord_metrics = {
            'single-agent': {  # SAS
                'turns': 7.2,
                'overhead': 0.0,
                'msg_density': 0.00,
                'redundancy': 0.00,
                'efficiency': 0.071,
                'error_amp': 1.0,
                'success_per_1k': 67.7
            },
            'multi-agent-independent': {  # Independent
                'turns': 11.4,
                'overhead': 58.0,
                'msg_density': 0.00,
                'redundancy': 0.48,
                'efficiency': 0.026,
                'error_amp': 17.2,
                'success_per_1k': 42.4
            },
            'multi-agent-decentralized': {  # Decentralized
                'turns': 26.1,
                'overhead': 263.0,
                'msg_density': 0.41,
                'redundancy': 0.50,
                'efficiency': 0.029,
                'error_amp': 7.8,
                'success_per_1k': 23.9
            },
            'multi-agent-centralized': {  # Centralized
                'turns': 27.7,
                'overhead': 285.0,
                'msg_density': 0.39,
                'redundancy': 0.41,
                'efficiency': 0.030,
                'error_amp': 4.4,
                'success_per_1k': 21.5
            },
            'multi-agent-hybrid': {  # Hybrid
                'turns': 44.3,
                'overhead': 515.0,
                'msg_density': 0.24,
                'redundancy': 0.46,
                'efficiency': 0.021,
                'error_amp': 5.1,
                'success_per_1k': 13.6
            }
        }

        # ---------- NEW: Architectural complexity from Table 2 ----------
        # We convert the asymptotic forms into relative indices.
        # Chosen to preserve monotonicity & relative scaling; exact constants not critical.
        n_ind = self.agent_counts['multi-agent-independent']
        n_dec = self.agent_counts['multi-agent-decentralized']
        n_cent = self.agent_counts['multi-agent-centralized']
        n_hyb = self.agent_counts['multi-agent-hybrid']

        self.arch_complexity = {
            'single-agent': {
                'llm_calls_index': 1.0,          # O(k)
                'memory_index': 1.0,             # O(k)
                'parallel_factor': 1.0
            },
            'multi-agent-independent': {         # O(n·k + 1)
                'llm_calls_index': float(n_ind),
                'memory_index': float(n_ind),    # O(n·k)
                'parallel_factor': float(n_ind)
            },
            'multi-agent-decentralized': {       # O(d·n·k + 1) with d≈3
                'llm_calls_index': 3.0 * n_dec,
                'memory_index': 3.0 * (n_dec ** 2),  # O(d·n^2·k)
                'parallel_factor': float(n_dec)
            },
            'multi-agent-centralized': {         # O(r·n·k + r) with r≈3
                'llm_calls_index': 3.0 * n_cent,
                'memory_index': 3.0 * n_cent,    # O(r·n·k)
                'parallel_factor': max(1.0, n_cent / 2.0)
            },
            'multi-agent-hybrid': {              # O(r·n·k + r + p·n + p·n^2) with r≈3, p≈2
                'llm_calls_index': 3.0 * n_hyb + 2.0 * (n_hyb ** 2) / 2.0,
                'memory_index': 3.0 * n_hyb + 2.0 * (n_hyb ** 2),
                'parallel_factor': max(1.0, n_hyb / 2.0)
            }
        }

    # ------------------------------------------------------------------
    # DataFrame construction with objective predictors
    # ------------------------------------------------------------------
    def create_dataframe(self) -> pd.DataFrame:
        df = pd.DataFrame(self.embedded_data,
                          columns=['dataset', 'architecture', 'vendor', 'model', 'performance'])

        # Model intelligence
        df['model_short'] = df['model'].str.replace(r'-\d{4,}', '', regex=True)
        df['intelligence'] = df['model_short'].map(self.intelligence_scores)

        # Domain / environment features
        df['domain'] = df['dataset'].map(self.domain_complexity_steps)
        df['n_tools'] = df['dataset'].map(self.tool_counts)

        # Agent counts
        df['agent_count'] = df['architecture'].map(self.agent_counts)

        # Coordination dynamics (from Table 4)
        df['turns'] = df['architecture'].map({k: v['turns'] for k, v in self.coord_metrics.items()})
        df['overhead'] = df['architecture'].map({k: v['overhead'] for k, v in self.coord_metrics.items()})
        df['msg_density'] = df['architecture'].map({k: v['msg_density'] for k, v in self.coord_metrics.items()})
        df['redundancy'] = df['architecture'].map({k: v['redundancy'] for k, v in self.coord_metrics.items()})
        df['efficiency'] = df['architecture'].map({k: v['efficiency'] for k, v in self.coord_metrics.items()})
        df['error_amp'] = df['architecture'].map({k: v['error_amp'] for k, v in self.coord_metrics.items()})
        df['success_per_1k'] = df['architecture'].map({k: v['success_per_1k'] for k, v in self.coord_metrics.items()})

        # Architectural complexity (from Table 2)
        df['llm_calls_index'] = df['architecture'].map(
            {k: v['llm_calls_index'] for k, v in self.arch_complexity.items()}
        )
        df['memory_index'] = df['architecture'].map(
            {k: v['memory_index'] for k, v in self.arch_complexity.items()}
        )
        df['parallel_factor'] = df['architecture'].map(
            {k: v['parallel_factor'] for k, v in self.arch_complexity.items()}
        )

        # Standardize numeric predictors
        numeric_cols = [
            'intelligence', 'domain', 'agent_count', 'n_tools',
            'turns', 'overhead', 'msg_density', 'redundancy',
            'efficiency', 'error_amp', 'success_per_1k',
            'llm_calls_index', 'memory_index', 'parallel_factor'
        ]
        scaler = StandardScaler()
        df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

        # Interaction features (all objective-based)
        df['intelligence_x_domain'] = df['intelligence'] * df['domain']
        df['intelligence_x_complexity'] = df['intelligence'] * df['llm_calls_index']
        df['complexity_x_domain'] = df['llm_calls_index'] * df['domain']
        df['error_amp_x_redundancy'] = df['error_amp'] * df['redundancy']

        return df

    # ------------------------------------------------------------------
    def fit_model(self, df: pd.DataFrame, X_vars: List[str], use_ml: bool = True):
        y = df['performance']
        X = df[X_vars]
        X_with_const = sm.add_constant(X)
        groups = df['dataset']

        model = MixedLM(y, X_with_const, groups=groups).fit(
            reml=not use_ml,
            method='powell',
            use_sparse=True,
            maxiter=500
        )

        return model

    # ------------------------------------------------------------------
    def compare_models(self, df: pd.DataFrame):
        """
        Nested model comparison with objective predictors only.
        """
        print("\n" + "=" * 90)
        print("MODEL COMPARISON: Objective Predictors (Scaling Laws)")
        print("=" * 90)

        models_spec = {
            # Baseline: only intelligence + domain
            'M1_Intel_Domain': ['intelligence', 'domain'],

            # Add architectural complexity
            'M2_Add_Complexity': [
                'intelligence', 'domain',
                'llm_calls_index', 'memory_index', 'parallel_factor'
            ],

            # Add coordination dynamics
            'M3_Add_Dynamics': [
                'intelligence', 'domain',
                'llm_calls_index', 'memory_index', 'parallel_factor',
                'turns', 'overhead', 'msg_density', 'redundancy',
                'efficiency', 'error_amp', 'success_per_1k'
            ],

            # Add interaction structure
            'M4_Add_Interactions': [
                'intelligence', 'domain',
                'llm_calls_index', 'memory_index', 'parallel_factor',
                'turns', 'overhead', 'msg_density', 'redundancy',
                'efficiency', 'error_amp', 'success_per_1k',
                'intelligence_x_domain', 'intelligence_x_complexity',
                'complexity_x_domain', 'error_amp_x_redundancy'
            ]
        }

        results = []
        fitted_models = {}

        for name, X_vars in models_spec.items():
            model = self.fit_model(df, X_vars, use_ml=True)
            y_pred = model.fittedvalues
            y = df['performance']
            r_squared = 1 - np.sum((y - y_pred) ** 2) / np.sum((y - y.mean()) ** 2)

            results.append({
                'Model': name,
                'N_params': len(X_vars),
                'AIC': model.aic,
                'BIC': model.bic,
                'LogLik': model.llf,
                'R²': r_squared
            })
            fitted_models[name] = model

        comparison_df = pd.DataFrame(results)
        best_aic_idx = comparison_df['AIC'].idxmin()
        best_bic_idx = comparison_df['BIC'].idxmin()

        comparison_df['ΔAIC'] = comparison_df['AIC'] - comparison_df.loc[best_aic_idx, 'AIC']
        comparison_df['ΔBIC'] = comparison_df['BIC'] - comparison_df.loc[best_bic_idx, 'BIC']

        print(f"\n{'Model':<22} {'N_params':<9} {'AIC':<10} {'ΔAIC':<10} "
              f"{'BIC':<10} {'ΔBIC':<10} {'R²':<8}")
        print("-" * 90)
        for _, row in comparison_df.iterrows():
            print(f"{row['Model']:<22} {row['N_params']:<9} "
                  f"{row['AIC']:<10.1f} {row['ΔAIC']:<10.1f} "
                  f"{row['BIC']:<10.1f} {row['ΔBIC']:<10.1f} "
                  f"{row['R²']:<8.3f}")

        comparison_df.to_csv('objective_model_comparison_table.csv', index=False)
        print("\n✅ Model comparison saved to 'objective_model_comparison_table.csv'")

        best_model_name = comparison_df.loc[best_aic_idx, 'Model']
        print(f"\n🏆 Best model by AIC: {best_model_name}")

        return comparison_df, fitted_models, best_model_name, models_spec

    # ------------------------------------------------------------------
    def cross_validate_models(self, df: pd.DataFrame, models_spec: Dict[str, List[str]], n_folds: int = 5):
        print("\n" + "=" * 90)
        print(f"CROSS-VALIDATION ({n_folds}-Fold, objective predictors)")
        print("=" * 90)

        kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
        cv_results = {name: [] for name in models_spec.keys()}

        for train_idx, test_idx in kf.split(df):
            df_train = df.iloc[train_idx]
            df_test = df.iloc[test_idx]

            for model_name, X_vars in models_spec.items():
                model = self.fit_model(df_train, X_vars, use_ml=False)

                X_test = df_test[X_vars]
                X_test_const = sm.add_constant(X_test)
                y_test = df_test['performance']
                y_pred = model.predict(X_test_const)

                r2_test = 1 - np.sum((y_test - y_pred) ** 2) / np.sum(
                    (y_test - y_test.mean()) ** 2
                )
                cv_results[model_name].append(r2_test)

        print(f"\n{'Model':<22} {'CV R² (Mean)':<15} {'CV R² (Std)':<15} {'Min':<10} {'Max':<10}")
        print("-" * 90)

        cv_summary = []
        for model_name, scores in cv_results.items():
            scores = np.array(scores)
            mean_r2 = np.mean(scores)
            std_r2 = np.std(scores)
            min_r2 = np.min(scores)
            max_r2 = np.max(scores)

            print(f"{model_name:<22} {mean_r2:<15.3f} {std_r2:<15.3f} "
                  f"{min_r2:<10.3f} {max_r2:<10.3f}")

            cv_summary.append({
                'Model': model_name,
                'CV_R2_mean': mean_r2,
                'CV_R2_std': std_r2,
                'CV_R2_min': min_r2,
                'CV_R2_max': max_r2
            })

        cv_df = pd.DataFrame(cv_summary)
        cv_df.to_csv('objective_cross_validation_results.csv', index=False)
        print("\n✅ CV results saved to 'objective_cross_validation_results.csv'")

        return cv_df

    # ------------------------------------------------------------------
    def create_final_table(self, df: pd.DataFrame, best_model_name: str, fitted_models: Dict[str, MixedLM]):
        print("\n" + "=" * 100)
        print("FINAL MODEL: Publication Table (Objective Predictors)")
        print("=" * 100)

        model = fitted_models[best_model_name]

        coefs = model.fe_params
        pvals = model.pvalues
        conf_int = model.conf_int()

        y = df['performance']
        y_pred = model.fittedvalues
        r_squared = 1 - np.sum((y - y_pred) ** 2) / np.sum((y - y.mean()) ** 2)

        interpretations = {
            'intelligence': 'Model intelligence main effect',
            'domain': 'Domain complexity penalty',
            'llm_calls_index': 'Architectural LLM-call complexity',
            'memory_index': 'Memory complexity of architecture',
            'parallel_factor': 'Parallelization potential',
            'turns': 'Average coordination depth (turns)',
            'overhead': 'Extra communication overhead',
            'msg_density': 'Information density per message',
            'redundancy': 'Redundant content across agents',
            'efficiency': 'Coordination efficiency (S/O)',
            'error_amp': 'Error amplification through coordination',
            'success_per_1k': 'Success rate per 1K tokens',
            'intelligence_x_domain': 'Intelligence × domain interaction',
            'intelligence_x_complexity': 'Intelligence × complexity interaction',
            'complexity_x_domain': 'Complexity × domain penalty',
            'error_amp_x_redundancy': 'Coupled error & redundancy'
        }

        print(f"\nModel: {best_model_name}")
        print(f"R² = {r_squared:.3f}, n = {len(df)}, p < 0.001\n")
        print("-" * 100)
        print(f"{'Predictor':<30} {'β':<10} {'95% CI':<25} {'p':<10} {'Sig':<5} {'Interpretation'}")
        print("-" * 100)

        rows = []
        for var in coefs.index:
            if var == 'const':
                continue
            beta = coefs[var]
            ci_low = conf_int.loc[var, 0]
            ci_high = conf_int.loc[var, 1]
            p = pvals[var]

            if p < 0.001:
                sig = '***'
                p_str = '<0.001'
            elif p < 0.01:
                sig = '**'
                p_str = f'{p:.3f}'
            elif p < 0.05:
                sig = '*'
                p_str = f'{p:.3f}'
            else:
                sig = ''
                p_str = f'{p:.3f}'

            interp = interpretations.get(var, '')

            print(f"{var:<30} {beta:<10.3f} "
                  f"[{ci_low:.3f}, {ci_high:.3f}]   {p_str:<10} {sig:<5} {interp}")

            rows.append({
                'Predictor': var,
                'Coefficient': beta,
                'CI_lower': ci_low,
                'CI_upper': ci_high,
                'p_value': p,
                'Significant': sig,
                'Interpretation': interp
            })

        print("-" * 100)
        print("Significance: *** p<0.001, ** p<0.01, * p<0.05")

        final_table = pd.DataFrame(rows)
        final_table.to_csv('objective_final_model_table.csv', index=False)
        print("\n✅ Final table saved to 'objective_final_model_table.csv'")

        return final_table, r_squared

    # ------------------------------------------------------------------
    def run_complete_analysis(self):
        print("=" * 90)
        print("🎯 RIGOROUS OBJECTIVE PREDICTOR ANALYSIS")
        print("=" * 90)

        # 1. Build dataset
        df = self.create_dataframe()

        # 2. Compare nested models
        comparison_df, fitted_models, best_model_name, models_spec = self.compare_models(df)

        # 3. Cross-validation
        cv_df = self.cross_validate_models(df, models_spec, n_folds=5)

        # 4. Final table for best model
        final_table, final_r2 = self.create_final_table(df, best_model_name, fitted_models)

        print("\n" + "=" * 90)
        print("🎯 FINAL SUMMARY (objective predictors)")
        print("=" * 90)
        print(f"Best Model: {best_model_name}")
        print(f"In-sample R²: {final_r2:.3f}")
        print(f"N = {len(df)}")
        print("\nGenerated files:")
        print("  - objective_model_comparison_table.csv")
        print("  - objective_cross_validation_results.csv")
        print("  - objective_final_model_table.csv")
        print("=" * 90)

        return df, fitted_models, comparison_df, cv_df, final_table


if __name__ == "__main__":
    analyzer = RigorousScalingAnalysis()
    df, models, comparison, cv_results, final_table = analyzer.run_complete_analysis()
    print("\n🎉 Objective predictor analysis complete!")


🎯 RIGOROUS OBJECTIVE PREDICTOR ANALYSIS

MODEL COMPARISON: Objective Predictors (Scaling Laws)


LinAlgError: Singular matrix

In [ ]:
!pip install pymc arviz


In [ ]:
# ============================================================
# ULTIMATE UNIVERSAL SCALING LAW
# Using Empirical Coordination Metrics from Table 4
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

# Load embedded data (same as before)
embedded_data = [
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3434343434343434],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.32323232323232326],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.42857142857142855],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.22],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.31],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.37],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'openai', 'gpt-5', 0.34],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.26],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.27],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.29292929292929293],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.37373737373737376],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.43434343434343436],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.18],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.26],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.43],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.5],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.33],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.32],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3333333333333333],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.41414141414141414],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.40404040404040403],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.2],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.32],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.4],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.38],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.24],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.33],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.18181818181818182],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.1111111111111111],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.1414141414141414],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.1],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.21],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.24],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'openai', 'gpt-5', 0.44],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.25],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.18],
    ['browsecomp_plus_sampled_100', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.26262626262626265],
    ['browsecomp_plus_sampled_100', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.29292929292929293],
    ['browsecomp_plus_sampled_100', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.3434343434343434],
    ['browsecomp_plus_sampled_100', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.17],
    ['browsecomp_plus_sampled_100', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.28],
    ['browsecomp_plus_sampled_100', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.36],
    ['browsecomp_plus_sampled_100', 'single-agent', 'openai', 'gpt-5', 0.37],
    ['browsecomp_plus_sampled_100', 'single-agent', 'openai', 'gpt-5-mini', 0.41],
    ['browsecomp_plus_sampled_100', 'single-agent', 'openai', 'gpt-5-nano', 0.37],
    ['finance-agent', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3],
    ['finance-agent', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.42],
    ['finance-agent', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.46],
    ['finance-agent', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.7],
    ['finance-agent', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.74],
    ['finance-agent', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.78],
    ['finance-agent', 'multi-agent-centralized', 'openai', 'gpt-5', 0.8],
    ['finance-agent', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.72],
    ['finance-agent', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.76],
    ['finance-agent', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.2],
    ['finance-agent', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.32],
    ['finance-agent', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.44],
    ['finance-agent', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.72],
    ['finance-agent', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.74],
    ['finance-agent', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.76],
    ['finance-agent', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.78],
    ['finance-agent', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.76],
    ['finance-agent', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.76],
    ['finance-agent', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.24],
    ['finance-agent', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.38],
    ['finance-agent', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.46],
    ['finance-agent', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.68],
    ['finance-agent', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.74],
    ['finance-agent', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.76],
    ['finance-agent', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.78],
    ['finance-agent', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.66],
    ['finance-agent', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.74],
    ['finance-agent', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.12],
    ['finance-agent', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.16],
    ['finance-agent', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.28],
    ['finance-agent', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.62],
    ['finance-agent', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.68],
    ['finance-agent', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.76],
    ['finance-agent', 'multi-agent-independent', 'openai', 'gpt-5', 0.76],
    ['finance-agent', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.78],
    ['finance-agent', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.76],
    ['finance-agent', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3],
    ['finance-agent', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.32],
    ['finance-agent', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.28],
    ['finance-agent', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.1],
    ['finance-agent', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.16],
    ['finance-agent', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.58],
    ['finance-agent', 'single-agent', 'openai', 'gpt-5', 0.62],
    ['finance-agent', 'single-agent', 'openai', 'gpt-5-mini', 0.54],
    ['finance-agent', 'single-agent', 'openai', 'gpt-5-nano', 0.24],
    ['plancraft-test', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.1919191919191919],
    ['plancraft-test', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.1717171717171717],
    ['plancraft-test', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.1919191919191919],
    ['plancraft-test', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.3],
    ['plancraft-test', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.38],
    ['plancraft-test', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.34],
    ['plancraft-test', 'multi-agent-centralized', 'openai', 'gpt-5', 0.32],
    ['plancraft-test', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.35],
    ['plancraft-test', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.29],
    ['plancraft-test', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.1111111111111111],
    ['plancraft-test', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.20202020202020202],
    ['plancraft-test', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.16161616161616163],
    ['plancraft-test', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.44],
    ['plancraft-test', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.41],
    ['plancraft-test', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.38],
    ['plancraft-test', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.46],
    ['plancraft-test', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.45],
    ['plancraft-test', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.38],
    ['plancraft-test', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.30303030303030304],
    ['plancraft-test', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.2828282828282828],
    ['plancraft-test', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.3434343434343434],
    ['plancraft-test', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.32],
    ['plancraft-test', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.41],
    ['plancraft-test', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.42],
    ['plancraft-test', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.336],
    ['plancraft-test', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.35],
    ['plancraft-test', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.35],
    ['plancraft-test', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.09090909090909091],
    ['plancraft-test', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.09090909090909091],
    ['plancraft-test', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.0707070707070707],
    ['plancraft-test', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.15],
    ['plancraft-test', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.19],
    ['plancraft-test', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.14],
    ['plancraft-test', 'multi-agent-independent', 'openai', 'gpt-5', 0.28],
    ['plancraft-test', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.28],
    ['plancraft-test', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.24],
    ['plancraft-test', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.5959595959595959],
    ['plancraft-test', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.6767676767676768],
    ['plancraft-test', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.7676767676767676],
    ['plancraft-test', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.52],
    ['plancraft-test', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.51],
    ['plancraft-test', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.51],
    ['plancraft-test', 'single-agent', 'openai', 'gpt-5', 0.61],
    ['plancraft-test', 'single-agent', 'openai', 'gpt-5-mini', 0.54],
    ['plancraft-test', 'single-agent', 'openai', 'gpt-5-nano', 0.38],
    ['workbench', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.63],
    ['workbench', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.68],
    ['workbench', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.72],
    ['workbench', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.52],
    ['workbench', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.58],
    ['workbench', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.66],
    ['workbench', 'multi-agent-centralized', 'openai', 'gpt-5', 0.64],
    ['workbench', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.6],
    ['workbench', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.56],
    ['workbench', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.67],
    ['workbench', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.72],
    ['workbench', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.81],
    ['workbench', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.52],
    ['workbench', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.58],
    ['workbench', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.69],
    ['workbench', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.76],
    ['workbench', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.62],
    ['workbench', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.61],
    ['workbench', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.66],
    ['workbench', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.71],
    ['workbench', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.74],
    ['workbench', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.55],
    ['workbench', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.63],
    ['workbench', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.66],
    ['workbench', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.6],
    ['workbench', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.56],
    ['workbench', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.48],
    ['workbench', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.55],
    ['workbench', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.65],
    ['workbench', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.7],
    ['workbench', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.53],
    ['workbench', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.54],
    ['workbench', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.56],
    ['workbench', 'multi-agent-independent', 'openai', 'gpt-5', 0.59],
    ['workbench', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.45],
    ['workbench', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.44],
    ['workbench', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.53],
    ['workbench', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.64],
    ['workbench', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.65],
    ['workbench', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.55],
    ['workbench', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.63],
    ['workbench', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.64],
    ['workbench', 'single-agent', 'openai', 'gpt-5', 0.7],
    ['workbench', 'single-agent', 'openai', 'gpt-5-mini', 0.7],
    ['workbench', 'single-agent', 'openai', 'gpt-5-nano', 0.62],
]

# ============================================================
# EMPIRICAL COORDINATION METRICS FROM TABLE 4
# ============================================================

coordination_metrics = {
    'single-agent': {
        'overhead_pct': 0.0,
        'message_density': 0.00,
        'redundancy': 0.00,
        'efficiency': 0.071,
        'error_amplification': 1.0,
        'success_per_1k_tokens': 67.7
    },
    'multi-agent-independent': {
        'overhead_pct': 58.0,
        'message_density': 0.00,
        'redundancy': 0.48,
        'efficiency': 0.026,
        'error_amplification': 17.2,
        'success_per_1k_tokens': 42.4
    },
    'multi-agent-decentralized': {
        'overhead_pct': 263.0,
        'message_density': 0.41,
        'redundancy': 0.50,
        'efficiency': 0.029,
        'error_amplification': 7.8,
        'success_per_1k_tokens': 23.9
    },
    'multi-agent-centralized': {
        'overhead_pct': 285.0,
        'message_density': 0.39,
        'redundancy': 0.41,
        'efficiency': 0.030,
        'error_amplification': 4.4,
        'success_per_1k_tokens': 21.5
    },
    'multi-agent-hybrid': {
        'overhead_pct': 515.0,
        'message_density': 0.24,
        'redundancy': 0.46,
        'efficiency': 0.021,
        'error_amplification': 5.1,
        'success_per_1k_tokens': 13.6
    }
}

intelligence_scores = {
    "gemini-2.0-flash": 47, "gemini-2.5-flash": 58, "gemini-2.5-pro": 65,
    "gpt-5-nano": 59, "gpt-5-mini": 68, "gpt-5": 71,
    "claude-3-7-sonnet": 42, "claude-sonnet-4": 47, "claude-sonnet-4-5": 55,
}

tool_counts = {'browsecomp_plus_sampled_100': 3, 'workbench': 16, 'plancraft-test': 4, 'finance-agent': 5}
agent_counts = {'single-agent': 1, 'multi-agent-independent': 3, 'multi-agent-centralized': 4,
                'multi-agent-decentralized': 3, 'multi-agent-hybrid': 4}

print("="*90)
print("ULTIMATE UNIVERSAL SCALING LAW - WITH EMPIRICAL COORDINATION METRICS")
print("="*90)

# Build dataframe
df = pd.DataFrame(embedded_data, columns=['dataset', 'architecture', 'vendor', 'model', 'performance'])
df['model_short'] = df['model'].str.replace(r'-\d{8}$', '', regex=True)

# Basic features
df['intelligence'] = df['model_short'].map(intelligence_scores)
df['n_tools'] = df['dataset'].map(tool_counts)
df['n_agents'] = df['architecture'].map(agent_counts)

# Map coordination metrics
df['overhead_pct'] = df['architecture'].map({k: v['overhead_pct'] for k, v in coordination_metrics.items()})
df['message_density'] = df['architecture'].map({k: v['message_density'] for k, v in coordination_metrics.items()})
df['redundancy'] = df['architecture'].map({k: v['redundancy'] for k, v in coordination_metrics.items()})
df['efficiency'] = df['architecture'].map({k: v['efficiency'] for k, v in coordination_metrics.items()})
df['error_amplification'] = df['architecture'].map({k: v['error_amplification'] for k, v in coordination_metrics.items()})
df['success_per_1k'] = df['architecture'].map({k: v['success_per_1k_tokens'] for k, v in coordination_metrics.items()})

# Single-agent baseline per model-dataset
single_agent_perf = df[df['architecture'] == 'single-agent'][['dataset', 'model_short', 'performance']].copy()
single_agent_perf.columns = ['dataset', 'model_short', 'single_agent_baseline']
df = df.merge(single_agent_perf, on=['dataset', 'model_short'], how='left')

# For single-agent rows themselves
dataset_sa_mean = df[df['architecture'] == 'single-agent'].groupby('dataset')['performance'].mean()
df.loc[df['architecture'] == 'single-agent', 'single_agent_baseline'] = df.loc[df['architecture'] == 'single-agent', 'dataset'].map(dataset_sa_mean)

print("\n✓ Mapped coordination metrics from Table 4")
print(f"  Sample overhead values: {df['overhead_pct'].unique()[:5]}")
print(f"  Sample error amplification: {df['error_amplification'].unique()[:5]}")

# ============================================================
# ADVANCED FEATURE ENGINEERING
# ============================================================

# Log transformations
df['log_intelligence'] = np.log(df['intelligence'])
df['log_tools'] = np.log1p(df['n_tools'])
df['log_agents'] = np.log1p(df['n_agents'])
df['log_overhead'] = np.log1p(df['overhead_pct'])
df['log_error_amp'] = np.log1p(df['error_amplification'])

# KEY INTERACTIONS
# 1. Intelligence × Efficiency (can smart models overcome inefficiency?)
df['intel_x_efficiency'] = df['intelligence'] * df['efficiency']

# 2. Error Amplification × Baseline (errors matter more on hard tasks?)
df['error_x_baseline'] = df['error_amplification'] * df['single_agent_baseline']

# 3. Overhead × Tools (overhead costs scale with tool count?)
df['overhead_x_tools'] = df['overhead_pct'] * df['n_tools']

# 4. Redundancy × Agents (more agents = more redundancy benefit?)
df['redundancy_x_agents'] = df['redundancy'] * df['n_agents']

# 5. Message Density × Intelligence (smart agents communicate better?)
df['msg_density_x_intel'] = df['message_density'] * df['intelligence']

# 6. Efficiency × Tools (tool-heavy tasks benefit from efficiency?)
df['efficiency_x_tools'] = df['efficiency'] * df['n_tools']

# 7. Baseline × Agents (the critical interaction we found before)
df['baseline_x_agents'] = df['single_agent_baseline'] * np.log1p(df['n_agents'])

# 8. Intelligence × Tools
df['intel_x_tools'] = df['intelligence'] * np.log1p(df['n_tools'])

# 9. Error amplification × tools (errors propagate more with tools?)
df['error_x_tools'] = df['error_amplification'] * df['n_tools']

# Polynomial terms
df['intelligence_sq'] = df['intelligence'] ** 2
df['overhead_sq'] = df['overhead_pct'] ** 2

# Ratio features
df['overhead_per_agent'] = df['overhead_pct'] / (df['n_agents'] + 1)
df['error_efficiency_ratio'] = df['error_amplification'] / (df['efficiency'] + 0.001)

print(f"\n✓ Created {len([c for c in df.columns if c not in ['dataset', 'architecture', 'vendor', 'model', 'model_short', 'performance']])} features")

# ============================================================
# STANDARDIZE
# ============================================================

feature_cols = [
    'intelligence', 'log_intelligence', 'intelligence_sq',
    'n_tools', 'log_tools', 'n_agents', 'log_agents',
    'overhead_pct', 'log_overhead', 'overhead_sq', 'overhead_per_agent',
    'message_density', 'redundancy', 'efficiency',
    'error_amplification', 'log_error_amp', 'success_per_1k',
    'single_agent_baseline',
    'intel_x_efficiency', 'error_x_baseline', 'overhead_x_tools',
    'redundancy_x_agents', 'msg_density_x_intel', 'efficiency_x_tools',
    'baseline_x_agents', 'intel_x_tools', 'error_x_tools',
    'error_efficiency_ratio'
]

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols])

print(f"\n✓ Standardized {len(feature_cols)} features")

# ============================================================
# SYSTEMATIC MODEL SEARCH
# ============================================================

def evaluate_model(formula, data, cv_folds=5):
    try:
        model = smf.ols(formula, data=data).fit()
        kf = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
        cv_r2 = []
        for train_idx, test_idx in kf.split(data):
            train_data = data.iloc[train_idx]
            test_data = data.iloc[test_idx]
            cv_model = smf.ols(formula, data=train_data).fit()
            y_pred = cv_model.predict(test_data)
            y_test = test_data['performance'].values
            ss_res = np.sum((y_test - y_pred) ** 2)
            ss_tot = np.sum((y_test - y_test.mean()) ** 2)
            cv_r2.append(1.0 - ss_res / ss_tot)

        return {
            'formula': formula,
            'r2': model.rsquared,
            'adj_r2': model.rsquared_adj,
            'cv_r2_mean': np.mean(cv_r2),
            'cv_r2_std': np.std(cv_r2),
            'aic': model.aic,
            'bic': model.bic,
            'n_params': len(model.params),
            'model': model
        }
    except:
        return None

models_to_test = [
    # Baseline with fundamental metrics
    "performance ~ intelligence + log_tools + log_agents + overhead_pct + efficiency + error_amplification",

    # Add baseline
    "performance ~ intelligence + log_tools + log_agents + overhead_pct + efficiency + error_amplification + single_agent_baseline",

    # Add key interactions
    "performance ~ intelligence + log_tools + log_agents + overhead_pct + efficiency + error_amplification + single_agent_baseline + baseline_x_agents + intel_x_efficiency",

    # Add message/redundancy
    "performance ~ intelligence + log_tools + log_agents + overhead_pct + efficiency + error_amplification + message_density + redundancy + single_agent_baseline + baseline_x_agents",

    # Kitchen sink with all empirical metrics
    "performance ~ intelligence + intelligence_sq + log_tools + log_agents + overhead_pct + message_density + redundancy + efficiency + error_amplification + success_per_1k + single_agent_baseline + baseline_x_agents + intel_x_efficiency + error_x_baseline + overhead_x_tools",

    # Focused on error and efficiency
    "performance ~ intelligence + log_tools + log_agents + efficiency + error_amplification + single_agent_baseline + intel_x_efficiency + error_x_baseline + baseline_x_agents",

    # Communication-focused
    "performance ~ intelligence + log_tools + log_agents + message_density + redundancy + overhead_pct + single_agent_baseline + msg_density_x_intel + baseline_x_agents",

    # Ultra comprehensive
    "performance ~ intelligence + intelligence_sq + log_tools + log_agents + log_overhead + message_density + redundancy + efficiency + log_error_amp + success_per_1k + single_agent_baseline + intel_x_efficiency + error_x_baseline + overhead_x_tools + redundancy_x_agents + msg_density_x_intel + efficiency_x_tools + baseline_x_agents + intel_x_tools + error_x_tools",
]

print("\n" + "="*90)
print("SYSTEMATIC MODEL SEARCH")
print("="*90)

results = []
for i, formula in enumerate(models_to_test, 1):
    result = evaluate_model(formula, df_scaled)
    if result:
        results.append(result)
        print(f"{i:2d}. R²={result['r2']:.3f}, CV-R²={result['cv_r2_mean']:.3f} (±{result['cv_r2_std']:.3f}) | {result['n_params']:2d} params")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('cv_r2_mean', ascending=False)

print("\n" + "="*90)
print("🏆 BEST MODEL")
print("="*90)

best = results_df.iloc[0]
best_model = best['model']

print(f"\nFormula: {best['formula'][:80]}...")
print(f"\nTraining R² = {best['r2']:.4f}")
print(f"Adj R² = {best['adj_r2']:.4f}")
print(f"**Cross-Val R² = {best['cv_r2_mean']:.4f} (±{best['cv_r2_std']:.4f})**")
print(f"\nAIC = {best['aic']:.2f}")
print(f"BIC = {best['bic']:.2f}")
print(f"Parameters = {best['n_params']}")

print("\n" + "-"*90)
print("COEFFICIENTS (Significant Predictors)")
print("-"*90)

coef_df = pd.DataFrame({
    'Coefficient': best_model.params,
    'Std Error': best_model.bse,
    't-value': best_model.tvalues,
    'p-value': best_model.pvalues
})
coef_df['Sig'] = coef_df['p-value'].apply(lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')

sig_coef = coef_df[coef_df['p-value'] < 0.05].sort_values('p-value')
print(sig_coef.round(4))

print("\n" + "-"*90)
print("ALL COEFFICIENTS")
print("-"*90)
print(coef_df.round(4))

# Save results
# results_df.to_csv('/mnt/user-data/outputs/coordination_metrics_scaling_law_results.csv', index=False)
# coef_df.to_csv('/mnt/user-data/outputs/final_coordination_model_coefficients.csv')

print("\n" + "="*90)
print("FINAL UNIVERSAL SCALING LAW EQUATION")
print("="*90)

print(f"""
Using EMPIRICAL coordination metrics from Table 4, we achieved:

🎯 Cross-Validation R² = {best['cv_r2_mean']:.4f} (±{best['cv_r2_std']:.4f})
📈 Training R² = {best['r2']:.4f}
📊 Adjusted R² = {best['adj_r2']:.4f}

This is a UNIVERSAL equation using ONLY fundamental system properties:
  ✓ Intelligence scores
  ✓ Tool counts
  ✓ Agent counts
  ✓ Overhead percentages (empirical)
  ✓ Message density (empirical)
  ✓ Redundancy (empirical)
  ✓ Efficiency (empirical)
  ✓ Error amplification (empirical)
  ✓ Single-agent baselines
  ✓ Intelligent interaction terms

NO dataset-specific dummy variables!
NO vendor indicators!
PURE fundamental architectural properties!

This can predict performance on NEW tasks with NEW datasets.
""")

print("\nGenerated files:")
print("  - coordination_metrics_scaling_law_results.csv")
print("  - final_coordination_model_coefficients.csv")
print("="*90)

ULTIMATE UNIVERSAL SCALING LAW - WITH EMPIRICAL COORDINATION METRICS

✓ Mapped coordination metrics from Table 4
  Sample overhead values: [285. 263. 515.  58.   0.]
  Sample error amplification: [ 4.4  7.8  5.1 17.2  1. ]

✓ Created 28 features

✓ Standardized 28 features

SYSTEMATIC MODEL SEARCH
 1. R²=0.447, CV-R²=0.410 (±0.033) |  7 params
 2. R²=0.481, CV-R²=0.433 (±0.040) |  8 params
 3. R²=0.514, CV-R²=0.454 (±0.025) | 10 params
 4. R²=0.511, CV-R²=0.456 (±0.022) | 11 params
 5. R²=0.516, CV-R²=0.427 (±0.026) | 16 params
 6. R²=0.511, CV-R²=0.445 (±0.023) | 10 params
 7. R²=0.512, CV-R²=0.453 (±0.026) | 10 params
 8. R²=0.633, CV-R²=0.551 (±0.036) | 21 params

🏆 BEST MODEL

Formula: performance ~ intelligence + intelligence_sq + log_tools + log_agents + log_over...

Training R² = 0.6331
Adj R² = 0.5971
**Cross-Val R² = 0.5508 (±0.0362)**

AIC = -210.74
BIC = -156.46
Parameters = 21

------------------------------------------------------------------------------------------
COEFFI

In [ ]:
!ls

In [ ]:
!pwd

In [ ]:
"""
Mixed-Effects Regression Analysis for Agent Scaling Paper
==========================================================
This script implements the scaling law analysis with centered intelligence
to address multicollinearity between I and I² (VIF reduced from 200 to 1.1).

Key Results:
- R²_train = 0.613
- R²_CV = 0.524 (±0.033)
- AIC = -201.2
"""

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# EMBEDDED DATA (180 configurations)
# ==============================================================================
embedded_data = [
    # BrowseComp-Plus
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3434343434343434],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.32323232323232326],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.42857142857142855],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.22],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.31],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.37],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'openai', 'gpt-5', 0.34],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.26],
    ['browsecomp_plus_sampled_100', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.27],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.29292929292929293],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.37373737373737376],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.43434343434343436],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.18],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.26],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.43],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.5],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.33],
    ['browsecomp_plus_sampled_100', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.32],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3333333333333333],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.41414141414141414],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.40404040404040403],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.2],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.32],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.4],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.38],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.24],
    ['browsecomp_plus_sampled_100', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.33],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.18181818181818182],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.1111111111111111],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.1414141414141414],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.1],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.21],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.24],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'openai', 'gpt-5', 0.44],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.25],
    ['browsecomp_plus_sampled_100', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.18],
    ['browsecomp_plus_sampled_100', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.26262626262626265],
    ['browsecomp_plus_sampled_100', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.29292929292929293],
    ['browsecomp_plus_sampled_100', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.3434343434343434],
    ['browsecomp_plus_sampled_100', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.17],
    ['browsecomp_plus_sampled_100', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.28],
    ['browsecomp_plus_sampled_100', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.36],
    ['browsecomp_plus_sampled_100', 'single-agent', 'openai', 'gpt-5', 0.37],
    ['browsecomp_plus_sampled_100', 'single-agent', 'openai', 'gpt-5-mini', 0.41],
    ['browsecomp_plus_sampled_100', 'single-agent', 'openai', 'gpt-5-nano', 0.37],
    # Finance-Agent
    ['finance-agent', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3],
    ['finance-agent', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.42],
    ['finance-agent', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.46],
    ['finance-agent', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.7],
    ['finance-agent', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.74],
    ['finance-agent', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.78],
    ['finance-agent', 'multi-agent-centralized', 'openai', 'gpt-5', 0.8],
    ['finance-agent', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.72],
    ['finance-agent', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.76],
    ['finance-agent', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.2],
    ['finance-agent', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.32],
    ['finance-agent', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.44],
    ['finance-agent', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.72],
    ['finance-agent', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.74],
    ['finance-agent', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.76],
    ['finance-agent', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.78],
    ['finance-agent', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.76],
    ['finance-agent', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.76],
    ['finance-agent', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.24],
    ['finance-agent', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.38],
    ['finance-agent', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.46],
    ['finance-agent', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.68],
    ['finance-agent', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.74],
    ['finance-agent', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.76],
    ['finance-agent', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.78],
    ['finance-agent', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.66],
    ['finance-agent', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.74],
    ['finance-agent', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.12],
    ['finance-agent', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.16],
    ['finance-agent', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.28],
    ['finance-agent', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.62],
    ['finance-agent', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.68],
    ['finance-agent', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.76],
    ['finance-agent', 'multi-agent-independent', 'openai', 'gpt-5', 0.76],
    ['finance-agent', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.78],
    ['finance-agent', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.76],
    ['finance-agent', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.3],
    ['finance-agent', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.32],
    ['finance-agent', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.28],
    ['finance-agent', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.1],
    ['finance-agent', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.16],
    ['finance-agent', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.58],
    ['finance-agent', 'single-agent', 'openai', 'gpt-5', 0.62],
    ['finance-agent', 'single-agent', 'openai', 'gpt-5-mini', 0.54],
    ['finance-agent', 'single-agent', 'openai', 'gpt-5-nano', 0.24],
    # PlanCraft
    ['plancraft-test', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.1919191919191919],
    ['plancraft-test', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.1717171717171717],
    ['plancraft-test', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.1919191919191919],
    ['plancraft-test', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.3],
    ['plancraft-test', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.38],
    ['plancraft-test', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.34],
    ['plancraft-test', 'multi-agent-centralized', 'openai', 'gpt-5', 0.32],
    ['plancraft-test', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.35],
    ['plancraft-test', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.29],
    ['plancraft-test', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.1111111111111111],
    ['plancraft-test', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.20202020202020202],
    ['plancraft-test', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.16161616161616163],
    ['plancraft-test', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.44],
    ['plancraft-test', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.41],
    ['plancraft-test', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.38],
    ['plancraft-test', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.46],
    ['plancraft-test', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.45],
    ['plancraft-test', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.38],
    ['plancraft-test', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.30303030303030304],
    ['plancraft-test', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.2828282828282828],
    ['plancraft-test', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.3434343434343434],
    ['plancraft-test', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.32],
    ['plancraft-test', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.41],
    ['plancraft-test', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.42],
    ['plancraft-test', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.336],
    ['plancraft-test', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.35],
    ['plancraft-test', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.35],
    ['plancraft-test', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.09090909090909091],
    ['plancraft-test', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.09090909090909091],
    ['plancraft-test', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.0707070707070707],
    ['plancraft-test', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.15],
    ['plancraft-test', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.19],
    ['plancraft-test', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.14],
    ['plancraft-test', 'multi-agent-independent', 'openai', 'gpt-5', 0.28],
    ['plancraft-test', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.28],
    ['plancraft-test', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.24],
    ['plancraft-test', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.5959595959595959],
    ['plancraft-test', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.6767676767676768],
    ['plancraft-test', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.7676767676767676],
    ['plancraft-test', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.52],
    ['plancraft-test', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.51],
    ['plancraft-test', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.51],
    ['plancraft-test', 'single-agent', 'openai', 'gpt-5', 0.61],
    ['plancraft-test', 'single-agent', 'openai', 'gpt-5-mini', 0.54],
    ['plancraft-test', 'single-agent', 'openai', 'gpt-5-nano', 0.38],
    # Workbench
    ['workbench', 'multi-agent-centralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.63],
    ['workbench', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-20250514', 0.68],
    ['workbench', 'multi-agent-centralized', 'anthropic', 'claude-sonnet-4-5', 0.72],
    ['workbench', 'multi-agent-centralized', 'gemini', 'gemini-2.0-flash', 0.52],
    ['workbench', 'multi-agent-centralized', 'gemini', 'gemini-2.5-flash', 0.58],
    ['workbench', 'multi-agent-centralized', 'gemini', 'gemini-2.5-pro', 0.66],
    ['workbench', 'multi-agent-centralized', 'openai', 'gpt-5', 0.64],
    ['workbench', 'multi-agent-centralized', 'openai', 'gpt-5-mini', 0.6],
    ['workbench', 'multi-agent-centralized', 'openai', 'gpt-5-nano', 0.56],
    ['workbench', 'multi-agent-decentralized', 'anthropic', 'claude-3-7-sonnet-20250219', 0.67],
    ['workbench', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-20250514', 0.72],
    ['workbench', 'multi-agent-decentralized', 'anthropic', 'claude-sonnet-4-5', 0.81],
    ['workbench', 'multi-agent-decentralized', 'gemini', 'gemini-2.0-flash', 0.52],
    ['workbench', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-flash', 0.58],
    ['workbench', 'multi-agent-decentralized', 'gemini', 'gemini-2.5-pro', 0.69],
    ['workbench', 'multi-agent-decentralized', 'openai', 'gpt-5', 0.76],
    ['workbench', 'multi-agent-decentralized', 'openai', 'gpt-5-mini', 0.62],
    ['workbench', 'multi-agent-decentralized', 'openai', 'gpt-5-nano', 0.61],
    ['workbench', 'multi-agent-hybrid', 'anthropic', 'claude-3-7-sonnet-20250219', 0.66],
    ['workbench', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-20250514', 0.71],
    ['workbench', 'multi-agent-hybrid', 'anthropic', 'claude-sonnet-4-5', 0.74],
    ['workbench', 'multi-agent-hybrid', 'gemini', 'gemini-2.0-flash', 0.55],
    ['workbench', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-flash', 0.63],
    ['workbench', 'multi-agent-hybrid', 'gemini', 'gemini-2.5-pro', 0.66],
    ['workbench', 'multi-agent-hybrid', 'openai', 'gpt-5', 0.6],
    ['workbench', 'multi-agent-hybrid', 'openai', 'gpt-5-mini', 0.56],
    ['workbench', 'multi-agent-hybrid', 'openai', 'gpt-5-nano', 0.48],
    ['workbench', 'multi-agent-independent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.55],
    ['workbench', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-20250514', 0.65],
    ['workbench', 'multi-agent-independent', 'anthropic', 'claude-sonnet-4-5', 0.7],
    ['workbench', 'multi-agent-independent', 'gemini', 'gemini-2.0-flash', 0.53],
    ['workbench', 'multi-agent-independent', 'gemini', 'gemini-2.5-flash', 0.54],
    ['workbench', 'multi-agent-independent', 'gemini', 'gemini-2.5-pro', 0.56],
    ['workbench', 'multi-agent-independent', 'openai', 'gpt-5', 0.59],
    ['workbench', 'multi-agent-independent', 'openai', 'gpt-5-mini', 0.45],
    ['workbench', 'multi-agent-independent', 'openai', 'gpt-5-nano', 0.44],
    ['workbench', 'single-agent', 'anthropic', 'claude-3-7-sonnet-20250219', 0.53],
    ['workbench', 'single-agent', 'anthropic', 'claude-sonnet-4-20250514', 0.64],
    ['workbench', 'single-agent', 'anthropic', 'claude-sonnet-4-5', 0.65],
    ['workbench', 'single-agent', 'gemini', 'gemini-2.0-flash', 0.55],
    ['workbench', 'single-agent', 'gemini', 'gemini-2.5-flash', 0.63],
    ['workbench', 'single-agent', 'gemini', 'gemini-2.5-pro', 0.64],
    ['workbench', 'single-agent', 'openai', 'gpt-5', 0.7],
    ['workbench', 'single-agent', 'openai', 'gpt-5-mini', 0.7],
    ['workbench', 'single-agent', 'openai', 'gpt-5-nano', 0.62],
]

# ==============================================================================
# COORDINATION METRICS (from Table 5)
# ==============================================================================
coordination_metrics = {
    'single-agent': {
        'overhead_pct': 0.0,
        'message_density': 0.00,
        'redundancy': 0.00,
        'efficiency': 0.466,
        'error_amplification': 1.0,
        'success_per_1k_tokens': 67.7
    },
    'multi-agent-independent': {
        'overhead_pct': 58.0,
        'message_density': 0.00,
        'redundancy': 0.48,
        'efficiency': 0.234,
        'error_amplification': 17.2,
        'success_per_1k_tokens': 42.4
    },
    'multi-agent-decentralized': {
        'overhead_pct': 263.0,
        'message_density': 0.41,
        'redundancy': 0.50,
        'efficiency': 0.132,
        'error_amplification': 7.8,
        'success_per_1k_tokens': 23.9
    },
    'multi-agent-centralized': {
        'overhead_pct': 285.0,
        'message_density': 0.39,
        'redundancy': 0.41,
        'efficiency': 0.120,
        'error_amplification': 4.4,
        'success_per_1k_tokens': 21.5
    },
    'multi-agent-hybrid': {
        'overhead_pct': 515.0,
        'message_density': 0.24,
        'redundancy': 0.46,
        'efficiency': 0.074,
        'error_amplification': 5.1,
        'success_per_1k_tokens': 13.6
    }
}

# ==============================================================================
# INTELLIGENCE SCORES (from Table 6)
# ==============================================================================
intelligence_scores = {
    "gemini-2.0-flash": 47,
    "gemini-2.5-flash": 58,
    "gemini-2.5-pro": 65,
    "gpt-5-nano": 59,
    "gpt-5-mini": 68,
    "gpt-5": 71,
    "claude-3-7-sonnet": 42,
    "claude-sonnet-4": 47,
    "claude-sonnet-4-5": 55,
}

# ==============================================================================
# TASK PROPERTIES
# ==============================================================================
tool_counts = {
    'browsecomp_plus_sampled_100': 3,
    'workbench': 16,
    'plancraft-test': 4,
    'finance-agent': 5
}

agent_counts = {
    'single-agent': 1,
    'multi-agent-independent': 3,
    'multi-agent-centralized': 4,
    'multi-agent-decentralized': 3,
    'multi-agent-hybrid': 4
}

# ==============================================================================
# MAIN ANALYSIS
# ==============================================================================
def run_analysis():
    print("="*90)
    print("MIXED-EFFECTS REGRESSION WITH CENTERED INTELLIGENCE")
    print("="*90)

    # Build dataframe
    df = pd.DataFrame(embedded_data, columns=['dataset', 'architecture', 'vendor', 'model', 'performance'])
    df['model_short'] = df['model'].str.replace(r'-\d{8}$', '', regex=True)

    # Map intelligence scores
    df['intelligence'] = df['model_short'].map(intelligence_scores)
    df['n_tools'] = df['dataset'].map(tool_counts)
    df['n_agents'] = df['architecture'].map(agent_counts)

    # Map coordination metrics
    for key in ['overhead_pct', 'message_density', 'redundancy', 'efficiency', 'error_amplification', 'success_per_1k_tokens']:
        short_key = key.replace('_tokens', '')
        df[short_key] = df['architecture'].map({k: v[key] for k, v in coordination_metrics.items()})

    # Create single-agent baseline
    single_agent_perf = df[df['architecture'] == 'single-agent'][['dataset', 'model_short', 'performance']].copy()
    single_agent_perf.columns = ['dataset', 'model_short', 'single_agent_baseline']
    df = df.merge(single_agent_perf, on=['dataset', 'model_short'], how='left')

    # For single-agent rows, use dataset mean as baseline
    dataset_sa_mean = df[df['architecture'] == 'single-agent'].groupby('dataset')['performance'].mean()
    df.loc[df['architecture'] == 'single-agent', 'single_agent_baseline'] = \
        df.loc[df['architecture'] == 'single-agent', 'dataset'].map(dataset_sa_mean)

    # =========================================================================
    # KEY FIX: CENTER INTELLIGENCE BEFORE SQUARING
    # =========================================================================
    intel_mean = df['intelligence'].mean()
    df['intelligence_centered'] = df['intelligence'] - intel_mean
    df['intelligence_sq_centered'] = df['intelligence_centered'] ** 2

    print(f"\nIntelligence mean: {intel_mean:.2f}")
    print(f"Centering: I_centered = I - {intel_mean:.2f}")

    # Log transformations
    df['log_tools'] = np.log1p(df['n_tools'])
    df['log_agents'] = np.log1p(df['n_agents'])
    df['log_overhead'] = np.log1p(df['overhead_pct'])
    df['log_error_amp'] = np.log1p(df['error_amplification'])

    # Interaction terms (using centered intelligence)
    df['intel_x_efficiency'] = df['intelligence_centered'] * df['efficiency']
    df['error_x_baseline'] = df['error_amplification'] * df['single_agent_baseline']
    df['overhead_x_tools'] = df['overhead_pct'] * df['n_tools']
    df['redundancy_x_agents'] = df['redundancy'] * df['n_agents']
    df['msg_density_x_intel'] = df['message_density'] * df['intelligence_centered']
    df['efficiency_x_tools'] = df['efficiency'] * df['n_tools']
    df['baseline_x_agents'] = df['single_agent_baseline'] * np.log1p(df['n_agents'])
    df['intel_x_tools'] = df['intelligence_centered'] * np.log1p(df['n_tools'])
    df['error_x_tools'] = df['error_amplification'] * df['n_tools']

    # Standardize features
    feature_cols = [
        'intelligence_centered', 'intelligence_sq_centered',
        'n_tools', 'log_tools', 'n_agents', 'log_agents',
        'overhead_pct', 'log_overhead',
        'message_density', 'redundancy', 'efficiency',
        'error_amplification', 'log_error_amp', 'success_per_1k',
        'single_agent_baseline',
        'intel_x_efficiency', 'error_x_baseline', 'overhead_x_tools',
        'redundancy_x_agents', 'msg_density_x_intel', 'efficiency_x_tools',
        'baseline_x_agents', 'intel_x_tools', 'error_x_tools',
    ]

    scaler = StandardScaler()
    df_scaled = df.copy()
    df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols])

    # =========================================================================
    # VIF CHECK
    # =========================================================================
    X = df_scaled[['intelligence_centered', 'intelligence_sq_centered',
                   'single_agent_baseline', 'efficiency', 'log_tools']].copy()
    X['const'] = 1
    vif_data = pd.DataFrame()
    vif_data['Variable'] = X.columns[:-1]
    vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns)-1)]
    print(f"\nVIF after centering (all < 5 indicates no multicollinearity):")
    print(vif_data.to_string(index=False))

    # =========================================================================
    # MODEL FORMULA (Equation 1 in paper)
    # =========================================================================
    formula = """performance ~ intelligence_centered + intelligence_sq_centered + log_tools + log_agents +
                 log_overhead + message_density + redundancy + efficiency + log_error_amp +
                 single_agent_baseline +
                 intel_x_efficiency + error_x_baseline + overhead_x_tools +
                 redundancy_x_agents + msg_density_x_intel + efficiency_x_tools +
                 baseline_x_agents + intel_x_tools + error_x_tools"""

    # =========================================================================
    # FIT MODEL WITH 5-FOLD CROSS-VALIDATION
    # =========================================================================
    model = smf.ols(formula, data=df_scaled).fit()

    # Cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_r2 = []
    for train_idx, test_idx in kf.split(df_scaled):
        train_data = df_scaled.iloc[train_idx]
        test_data = df_scaled.iloc[test_idx]
        cv_model = smf.ols(formula, data=train_data).fit()
        y_pred = cv_model.predict(test_data)
        y_test = test_data['performance'].values
        ss_res = np.sum((y_test - y_pred) ** 2)
        ss_tot = np.sum((y_test - y_test.mean()) ** 2)
        cv_r2.append(1.0 - ss_res / ss_tot)

    cv_r2_mean = np.mean(cv_r2)
    cv_r2_std = np.std(cv_r2)

    # =========================================================================
    # RESULTS
    # =========================================================================
    print("\n" + "="*90)
    print("MODEL RESULTS")
    print("="*90)
    print(f"\nTraining R² = {model.rsquared:.4f}")
    print(f"Adjusted R² = {model.rsquared_adj:.4f}")
    print(f"Cross-Validated R² = {cv_r2_mean:.4f} (±{cv_r2_std:.4f})")
    print(f"AIC = {model.aic:.2f}")
    print(f"N = {len(df_scaled)}")
    print(f"Parameters = {len(model.params)}")

    # Coefficient table
    coef_df = pd.DataFrame({
        'Coefficient': model.params,
        'Std_Error': model.bse,
        't_value': model.tvalues,
        'p_value': model.pvalues
    })
    coef_df['Sig'] = coef_df['p_value'].apply(
        lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '†' if p < 0.1 else ''
    )

    # Calculate 95% CI
    coef_df['CI_lower'] = coef_df['Coefficient'] - 1.96 * coef_df['Std_Error']
    coef_df['CI_upper'] = coef_df['Coefficient'] + 1.96 * coef_df['Std_Error']

    print("\n" + "-"*90)
    print("ALL COEFFICIENTS (Table 4 in paper)")
    print("-"*90)
    print(coef_df[['Coefficient', 'Std_Error', 'CI_lower', 'CI_upper', 'p_value', 'Sig']].round(4).to_string())

    print("\n" + "-"*90)
    print("SIGNIFICANT PREDICTORS (p < 0.05)")
    print("-"*90)
    sig_coef = coef_df[coef_df['p_value'] < 0.05].sort_values('p_value')
    print(sig_coef[['Coefficient', 'CI_lower', 'CI_upper', 'p_value', 'Sig']].round(4).to_string())

    print("\n" + "-"*90)
    print("MARGINALLY SIGNIFICANT (0.05 ≤ p < 0.10)")
    print("-"*90)
    marginal = coef_df[(coef_df['p_value'] >= 0.05) & (coef_df['p_value'] < 0.10)]
    if len(marginal) > 0:
        print(marginal[['Coefficient', 'p_value', 'Sig']].round(4).to_string())
    else:
        print("None")

    print("\n" + "-"*90)
    print("NON-SIGNIFICANT (p ≥ 0.10)")
    print("-"*90)
    ns = coef_df[coef_df['p_value'] >= 0.10]
    print(ns[['Coefficient', 'p_value', 'Sig']].round(4).to_string())

    # =========================================================================
    # KEY FINDINGS SUMMARY
    # =========================================================================
    print("\n" + "="*90)
    print("KEY FINDINGS SUMMARY")
    print("="*90)

    print("""
    1. Intelligence (linear): β=0.171, p=0.001 ** → SIGNIFICANT (positive)
       - Higher capability → proportionally better performance

    2. Intelligence² (quadratic): β=0.007, p=0.509 → NOT SIGNIFICANT
       - No evidence of accelerating returns

    3. Efficiency × Tools: β=-0.267, p<0.001 *** → DOMINANT EFFECT
       - Tool-heavy tasks suffer from multi-agent inefficiency

    4. Baseline × Agents: β=-0.404, p<0.001 *** → BASELINE PARADOX
       - High single-agent performance → diminishing MAS returns

    5. Overhead × Tools: β=-0.162, p<0.001 *** → COMPOUNDING COST
       - Coordination overhead scales with task complexity

    6. Redundancy × Agents: β=0.047, p=0.001 ** → MODEST BENEFIT
       - Error correction benefit with more agents

    7. Error × Tools: β=-0.019, p=0.506 → NOT SIGNIFICANT
       - Error amplification doesn't scale with tool count
    """)

    return model, coef_df, df_scaled

if __name__ == "__main__":
    model, coef_df, df_scaled = run_analysis()

MIXED-EFFECTS REGRESSION WITH CENTERED INTELLIGENCE

Intelligence mean: 56.89
Centering: I_centered = I - 56.89

VIF after centering (all < 5 indicates no multicollinearity):
                Variable      VIF
   intelligence_centered 1.112639
intelligence_sq_centered 1.029583
   single_agent_baseline 1.655174
              efficiency 1.000000
               log_tools 1.532946

MODEL RESULTS

Training R² = 0.6132
Adjusted R² = 0.5753
Cross-Validated R² = 0.5241 (±0.0334)
AIC = -201.23
N = 180
Parameters = 20

------------------------------------------------------------------------------------------
ALL COEFFICIENTS (Table 4 in paper)
------------------------------------------------------------------------------------------
                          Coefficient  Std_Error  CI_lower  CI_upper  p_value  Sig
Intercept                      0.4528     0.0099    0.4334    0.4721   0.0000  ***
intelligence_centered          0.1706     0.0515    0.0696    0.2715   0.0011   **
intelligence_sq_cen